# How much faster is the vectorized LTF scheduler than the iterative one across plan sizes?

The scheduler runs before any spectral computation and fixes every bin's frequency, resolution, segment length, number of averages, and segment starts. The iterative `ltf_plan` steps from one frequency to the next in a Python loop, so its cost grows with the number of frequencies and with the record length that the segment starts must cover. `vectorized_ltf_plan`, the `SpectrumAnalyzer` default, evaluates the LTF rules once on a dense logarithmic grid of $10\,J_{\rm des}$ candidate frequencies and then walks that grid with $r_j = f_{j+1} - f_j$. Its docstring describes the result as nearly identical to the iterative plan, with minor differences from the discrete grid. A speedup is useful only if the plan stays the same, and this notebook measures time alone.

Both functions are called directly, with no spectrum computed, so no spectral convention enters. Four scenarios vary the plan size at $f_s = 1000$ Hz, overlap 0.5, and `bmin` 1.5: $N = 10^5$ with `Jdes` 500 (`Lmin` 32, `Kdes` 50); $N = 10^6$ with `Jdes` 2000 and 10000; and $N = 10^7$ with `Jdes` 2000 (`Lmin` 128 and `Kdes` 100 for the last three). Each scheduler runs 10 times per scenario under `timeit.repeat`, and the mean time is recorded. The reference is the iterative `ltf_plan`, and the speedup is its mean time divided by the vectorized mean time.

In [ ]:
import timeit
import numpy as np
import pandas as pd
import math
import sys
import logging
import matplotlib.pyplot as plt
from IPython.display import display, HTML  # safe table display without .style

from speckit.schedulers import vectorized_ltf_plan, ltf_plan

In [ ]:
test_scenarios = [
    {"description": "Small", "params": {'N': 100_000, 'Jdes': 500, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 32, 'Kdes': 50}},
    {"description": "Typical", "params": {'N': 1_000_000, 'Jdes': 2000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
    {"description": "High-Res (Jdes)", "params": {'N': 1_000_000, 'Jdes': 10000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
    {"description": "Long Data (N)", "params": {'N': 10_000_000, 'Jdes': 2000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
]
num_runs = 10
results_data = []

print("Comparing Performance of LTF Schedulers (running...)")

vectorized_ltf_plan(**(test_scenarios[0]["params"]))

for scenario in test_scenarios:
    params = scenario["params"]
    
    # Use globals() in notebook environment for timeit
    iter_times = timeit.repeat(lambda: ltf_plan(**params), number=1, repeat=num_runs)
    vec_times = timeit.repeat(lambda: vectorized_ltf_plan(**params), number=1, repeat=num_runs)

    iter_avg_time = np.mean(iter_times)
    vec_avg_time = np.mean(vec_times)
    speedup = iter_avg_time / vec_avg_time if vec_avg_time > 0 else float('inf')
    
    results_data.append({
        "Scenario": scenario["description"],
        "Iterative Time (s)": iter_avg_time,
        "Vectorized Time (s)": vec_avg_time,
        "Speedup (x)": speedup
    })

In [ ]:
results_df = pd.DataFrame(results_data)

# Make a formatted copy for display
fmt_df = results_df.copy()
fmt_df["Iterative Time (s)"] = fmt_df["Iterative Time (s)"].map(lambda v: f"{v:.4f}")
fmt_df["Vectorized Time (s)"] = fmt_df["Vectorized Time (s)"].map(lambda v: f"{v:.4f}")
fmt_df["Speedup (x)"] = fmt_df["Speedup (x)"].map(lambda v: f"{v:.2f}x")

caption_html = "<h3>Performance Benchmark Results</h3>"
table_html = fmt_df.to_html(index=False, escape=False)  # pure pandas HTML, no jinja2
display(HTML(caption_html + table_html))

In [ ]:
labels = results_df['Scenario']
iter_times = results_df['Iterative Time (s)']
vec_times = results_df['Vectorized Time (s)']
speedups = results_df['Speedup (x)']
x = np.arange(len(labels))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fig.suptitle('LTF Scheduler Performance Comparison', fontsize=16)

# --- Plot 1: Execution Time ---
rects1 = ax1.bar(x - width/2, iter_times, width, label='Iterative (ltf_plan)', color='skyblue')
rects2 = ax1.bar(x + width/2, vec_times, width, label='Vectorized', color='coral')
ax1.set_ylabel('Execution Time (seconds)')
ax1.set_title('Execution Time per Scenario (Log Scale)')
ax1.set_yscale('log')
ax1.legend()
ax1.bar_label(rects1, padding=3, fmt='%.3f')
ax1.bar_label(rects2, padding=3, fmt='%.3f')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# --- Plot 2: Speedup Factor ---
rects3 = ax2.bar(x, speedups, color='mediumseagreen')
ax2.set_ylabel('Speedup Factor (x)')
ax2.set_title('Speedup Factor of Vectorized vs. Iterative')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=0)
ax2.bar_label(rects3, padding=3, fmt='%.1fx')
ax2.axhline(1, color='grey', linestyle='--', lw=1) # Add a line at 1x for reference
ax2.grid(axis='y', linestyle='--', alpha=0.7)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Reading the result

The first cell prints a start message and runs the timings. The second shows a table of mean iterative time, mean vectorized time, and speedup per scenario. The figure repeats the table: execution times as paired bars on a log scale, and speedup factors as bars with a dashed line at 1×.

The comparison that decides the notebook is the speedup against 1, and how it changes between scenarios: with `Jdes` at fixed $N$ (Typical against High-Res), and with $N$ at fixed `Jdes` (Typical against Long Data). The source fixes no expected speedup, and the numbers depend on the machine.

Limits in the code: one warm-up call precedes the timings, for the vectorized scheduler only. Each time is a mean over 10 runs, so a single slow run raises it. Both timings include building the per-frequency segment starts. The notebook does not compare the plans themselves. The vectorized walk takes `r`, `L`, and `K` from the first grid point at or above the current frequency, so its plan can differ from `ltf_plan` by up to one grid step; the scheduler comparison notebook checks the plans.